# 多个国家的校对 欧盟

# 函数定义
---

In [30]:
# 函数一、二、三定义
import pandas as pd 
import numpy as np

# 函数一
# 规范化fao数据，注意单位
# 输入所有年份的fao原始数据,输出分年份规范化好的fao数据
def fao_standard(fao_data,params,target_path):
    # params是字典，键是fao数据的动物种类名称，值是键对应的要转化的种类名称，所有值是列表则证明要将这两项相加
    # 确保数值列可以做四则运算
    value_name = "Value" if "Value" in fao_data.columns else "value"
    fao_data[value_name] = fao_data[value_name].astype(str)
    fao_data[value_name] = fao_data[value_name].str.replace(',', '')
    fao_data[value_name] = pd.to_numeric(fao_data[value_name], errors='coerce')

    # 单位转化
    fao_data.loc[fao_data['Unit'] == '1000 An', 'Value'] *= 1000
    fao_data.loc[fao_data['Unit'] == '1000 An', 'Unit'] = 'An'

    for year,year_group in fao_data.groupby("Year"):
        group = pd.DataFrame()
        # 一年一年,在此基础上再一个一个国家来
        for country,country_group in year_group.groupby("Area"):

            # fao_data数据只有Item列需要,用replace替换
            for item in params:

                if isinstance(item,str):

                    # 首先确保item在列的值里面
                    if item in list(country_group["Item"]):
                        country_group["Item"] = country_group['Item'].replace(item,params[item])

                elif isinstance(item,tuple):

                    # 先相加，按照年份，然后形成新的一个值
                    for i in range(1,len(item)):
                        if (item[i] in list(country_group["Item"])) and (item[0] in list(country_group["Item"])) :
                            country_group.loc[country_group["Item"]==item[0],value_name] += country_group[country_group['Item']==item[i]][value_name].values[0]
                    country_group["Item"] = country_group['Item'].replace(item[0],params[item]) 

                else:
                    print("params输入格式错误,错误的键为{}".format(item))
            # 国家合并在一起
            group = pd.concat([group,country_group],axis=0)

        # 保存
        group.to_csv(target_path+str(year)+".csv")


# 函数二
# 与FAO数据校对
# 欧盟多个国家的校对
def cal_r(data,data_fao,data_country_colname="country",data_fao_item_colname="item"):
    # data是之前已经整理好的数据，现在要被校对
    # data_fao是fao上的国家总量
    # data_country是data里面的国家列名
    # data_fao_item是fao里面的动物种类列名
    Year = data['year'].iloc[3]
    data['标记'] = ""

    # 首先筛选出动物种类
    animal_species = data_fao[data_fao_item_colname].unique() 
    value_name = "Value" if "Value" in data_fao.columns else "value"
    data_fao[value_name] = data_fao[value_name].astype(str)
    data_fao[value_name] = data_fao[value_name].str.replace(',', '')
    data_fao[value_name] = pd.to_numeric(data_fao[value_name], errors='coerce')

    # 筛选出国家
    country_species = data[data_country_colname].unique()

    data_only_state = data[data['county'].isna() & data['state'].notna()] # 取出州的
    # 仅保留县
    data.dropna(subset=['county','state'],inplace=True)
    r = data.iloc[:,0:6] # 存储比例
    r['year'] = Year
    # data = data[data['county'].notna() & data['state'].notna()] # 仅保留县

    for animal in animal_species:
        
        # 首先判断这个动物种类是否在data里面
        if animal in data.columns:
            r[animal] = ''
            # 首先转化列里面数值保证能进行四则运算
            r[animal] = r[animal].astype(str)
            r[animal] = r[animal].str.replace(',', '')  # remove commas
            r[animal] = pd.to_numeric(r[animal], errors='coerce').fillna(0)

            # 首先转化列里面数值保证能进行四则运算
            data[animal] = data[animal].astype(str)
            data[animal] = data[animal].str.replace(',', '')  # remove commas
            data[animal] = pd.to_numeric(data[animal], errors='coerce')
            # data[animal] *= 1000 # 单位转化

            for country in country_species:
                # 计算国家内各县的总量(注意不要加重复了)
                country_total = data_only_state[data_only_state[data_country_colname] == country][animal].sum() # 总量应该从州取
                if country_total==0:
                    # 如果总量等于0，则从县取
                    country_total = data[data[data_country_colname] == country][animal].sum() # 总量应该从县取
                    
                # 如果国家总量大于0，则根据占比分配FAO数据
                if country_total > 0:
                    if data[data[data_country_colname] == country][animal].sum() > 0.1: # 确保县不是空值
                            r.loc[r[data_country_colname]==country,animal] = (data[animal] / country_total) 
    return r



# 函数三
# 与FAO数据校对
# 欧盟多个国家的校对
def single_proofread_r(data,data_fao,r,data_country_colname="country",data_fao_item_colname="item"):
    # data是之前已经整理好的数据，现在要被校对
    # data_fao是fao上的国家总量
    # data_country是data里面的国家列名
    # data_fao_item是fao里面的动物种类列名
    # r是所有的历史比例
    Year = data['year'].iloc[3]
    count_zero = 0
    count_error = 0
    data['标记'] = ""

    # 首先筛选出动物种类
    animal_species = data_fao[data_fao_item_colname].unique() 
    value_name = "Value" if "Value" in data_fao.columns else "value"
    data_fao[value_name] = data_fao[value_name].astype(str)
    data_fao[value_name] = data_fao[value_name].str.replace(',', '')
    data_fao[value_name] = pd.to_numeric(data_fao[value_name], errors='coerce')

    # 筛选出国家
    country_species = data[data_country_colname].unique()

    data_only_state = data[data['county'].isna() & data['state'].notna()] # 取出州的
    # 仅保留县
    data.dropna(subset=['county','state'],inplace=True)
    # data = data[data['county'].notna() & data['state'].notna()] # 仅保留县

    # 在这里整合处理r值缺失的逻辑
    for country in country_species:
        for animal in animal_species:
            if animal in data.columns:

                # 对于每个国家，检查和处理r值缺失的情况
                filtered_r = r[(r[data_country_colname] == country) & (r['year'] == Year)]
                counties = data[data[data_country_colname] == country]['county'].unique()  # 假设县的信息在data中

                # 找出缺失r值的县
                missing_counties = [county for county in counties if county not in filtered_r['county'].unique()]

                # 如果有缺失的县
                if missing_counties:
                    # 随机生成r值，这里简化处理，假设等分给每个缺失的县
                    missing_r_values = np.random.dirichlet(np.ones(len(missing_counties)), size=1).flatten()
                    
                    # 创建缺失县的r值DataFrame
                    missing_r_df = pd.DataFrame({
                        'county': missing_counties,
                        data_country_colname: country,
                        'year': Year,
                        animal: missing_r_values
                    })
                    # 将新生成的r值DataFrame添加到原始DataFrame
                    r = pd.concat([r, missing_r_df], ignore_index=True)

                # 确保所有r值的总和等于1（可以根据需要进行调整）
                total_r = r[r[data_country_colname] == country][animal].sum()
                r.loc[r[data_country_colname] == country, animal] /= total_r

    Data_cols = data.columns[7:]
    for animal in Data_cols:
        data[animal] = data[animal].astype(str)
        data[animal] = data[animal].str.replace(',', '')  # remove commas
        data[animal] = pd.to_numeric(data[animal], errors='coerce')
        data[animal] = data[animal]*1000 # 单位统一 农作物用！！！

    for animal in animal_species:
        
        # 首先判断这个动物种类是否在data里面
        if animal in data.columns:
            # 首先转化列里面数值保证能进行四则运算
            r[animal] = r[animal].astype(str)
            r[animal] = r[animal].str.replace(',', '')  # remove commas
            r[animal] = pd.to_numeric(r[animal], errors='coerce').fillna(0)

            # 首先转化列里面数值保证能进行四则运算
            data[animal] = data[animal].astype(str)
            data[animal] = data[animal].str.replace(',', '')  # remove commas
            data[animal] = pd.to_numeric(data[animal], errors='coerce')

            for country in country_species:
                proportions = [0]

                # 获取FAO数据
                fao_value = data_fao[(data_fao[data_fao_item_colname] == animal) & (data_fao['Area'] == country)][value_name]
                if not fao_value.empty:
                    if pd.to_numeric(fao_value.iloc[0], errors='coerce') >= 0:
                        fao_value = fao_value.iloc[0]

                        # 计算国家内各县的总量(注意不要加重复了)
                        country_total = data_only_state[data_only_state[data_country_colname] == country][animal].sum() # 总量应该从州取
                        if country_total==0:
                            # 如果总量等于0，则从县取
                            country_total = data[data[data_country_colname] == country][animal].sum() # 总量应该从县取

                        # 首先计算data里面每个县占比
                        if country_total != 0:
                            proportions = data[data[data_country_colname] == country][animal] / country_total

                        # if abs(fao_value-country_total) > 0.2*max(fao_value,country_total):
                            # 标记一下这一行，然后跳过,区分是有数据还是没有数据导致的
                        count_error += 1
                            # if country_total==0 and fao_value != 0:
                        count_zero += 1
                                # 用fao_value代替country_total,country_total为0的化，国家里面的县也会为零，那这些县按照历史比例分配下去
                                # 找出最近的年份，其中 'animal' 列的值的总和大于等于 0
                                # for offset in range(1, max(Year - 1980, 2021 - Year) + 1):
                                #     recent_year = Year - offset if Year - offset >= 1980 else Year + offset
                                #     if recent_year <= 2021 and r[(r[data_country_colname]==country) & (r['year']==recent_year)][animal].sum() >= 0:
                        count = data.loc[data[data_country_colname] == country, animal].shape[0]
                                # 生成 n 个随机数
                        random_numbers = np.random.random(size=count)

                                # 将这些数除以它们的总和以确保它们加起来等于 1
                        normalized_numbers = random_numbers / random_numbers.sum()

                        # 创建一个 pandas Series
                        series = pd.Series(normalized_numbers)
                        
                        if sum(proportions)!=0:

                            series_values = (proportions * fao_value).values

                            data.loc[data[data_country_colname] == country, animal] = series_values

                        else:
                            series_values = (series * fao_value).values
                            
                            if(sum(series_values) != 0):
                                data.loc[data[data_country_colname] == country, animal] = series_values
                                        # break
                        # else:
                        #     data.loc[data[data_country_colname] == country, "标记"] += "种类："+animal+","+"FAO总量："+str(fao_value)+","+"国家总量："+str(country_total)+"; "

                            # print("国家为{}，种类为{}的fao总量为{}，对应国家数据总量{}".format(country,animal,fao_value,country_total))
                        


                        # 如果国家总量大于0，则根据占比分配FAO数据
                        # if country_total > 0:
                        #     if(country=='France'):
                        #         value = ((data[animal] / country_total) * fao_value)
                        #         print("Value为{},fao_value为{},动物为{}".format(value,fao_value,animal))
                        #         data.loc[data[data_country_colname] == country, animal] = value

                        #         print(data.loc[data[data_country_colname] == 'France', animal])

                            # data.loc[data[data_country_colname] == country, animal] = r[(r[data_country_colname]==country) & (r['year']==recent_year)][animal] * fao_value
                        # 大于零的情况可能是州不为0 ，但是县为0，所有当县为0 ，则总量要用之前的比例进行分配
                        # if data[data[data_country_colname] == country][animal].sum()==0:
                            # 找出最近的年份，其中 'animal' 列的值的总和大于等于 0
                            # for offset in range(1, max(Year - 1980, 2021 - Year) + 1):
                            #     recent_year = Year - offset if Year - offset >= 1980 else Year + offset
                            #     if recent_year <= 2021 and r[(r[data_country_colname]==country) & (r['year']==recent_year)][animal].sum() >= 0:
                            #         print("正常的r：{}".format(r[(r[data_country_colname]==country) & (r['year']==recent_year)][animal]))
                            #         data.loc[data[data_country_colname] == country, animal] = r[(r[data_country_colname]==country) & (r['year']==recent_year)][animal]  * fao_value
                            #         break
                                
                            # else:
                                # print("正常的r：{}".format((data[animal] / country_total)))
                            # value = ((data[animal] / country_total) * fao_value)
                            # data.loc[data[data_country_colname] == country, animal] = value
    # print("缺失占比为{}".format(count_zero/count_error))
    return data


# 函数四 单位转化
# 转化原始数据单位，确保与FAO统一
def unit_conversion(data,params):
    # params是字典，键是列名一部分(如area)，值是转化比例scale
    for col_name in params.keys():
        for item in data.columns:
            if isinstance(col_name,str):
                if col_name in item:
                    data[item] = data[item].astype(str)
                    data[item] = data[item].str.replace(',', '')
                    data[item] = pd.to_numeric(data[item], errors='coerce')
                    data[item] *= params[col_name]
            elif isinstance(col_name,tuple):
                for i in range(len(col_name)):
                    if col_name[i] in item:
                        data[item] = data[item].astype(str)
                        data[item] = data[item].str.replace(',', '')
                        data[item] = pd.to_numeric(data[item], errors='coerce')
                        data[item] *= params[col_name]
            else:
                print("params输入格式错误,错误的键为{}".format(item))

    return data



In [ ]:
                                
                            # else:
                                # print("正常的r：{}".format((data[animal] / country_total)))
                            # value = ((data[animal] / country_total) * fao_value)
                            # data.loc[data[data_country_colname] == country, animal] = value
    # print("缺失占比为{}".format(count_zero/count_error))
    return data


# 函数四 单位转化
# 转化原始数据单位，确保与FAO统一
def unit_conversion(data,params):
    # params是字典，键是列名一部分(如area)，值是转化比例scale
    for col_name in params.keys():
        for item in data.columns:
            if isinstance(col_name,str):
                if col_name in item:
                    data[item] = data[item].astype(str)
                    data[item] = data[item].str.replace(',', '')
                    data[item] = pd.to_numeric(data[item], errors='coerce')
                    data[item] *= params[col_name]
            elif isinstance(col_name,tuple):
                for i in range(len(col_name)):
                    if col_name[i] in item:
                        data[item] = data[item].astype(str)
                        data[item] = data[item].str.replace(',', '')
                        data[item] = pd.to_numeric(data[item], errors='coerce')
                        data[item] *= params[col_name]
            else:
                print("params输入格式错误,错误的键为{}".format(item))

    return data



In [ ]:
animal] = r[(r[data_country_colname]==country) & (r['year']==recent_year)][animal]  * fao_value
                            #         break
                                
                            # else:
                                # print("正常的r：{}".format((data[animal] / country_total)))
                            # value = ((data[animal] / country_total) * fao_value)
                            # data.loc[data[data_country_colname] == country, animal] = value
    # print("缺失占比为{}".format(count_zero/count_error))
    return data


# 函数四 单位转化
# 转化原始数据单位，确保与FAO统一
def unit_conversion(data,params):
    # params是字典，键是列名一部分(如area)，值是转化比例scale
    for col_name in params.keys():
        for item in data.columns:
            if isinstance(col_name,str):
                if col_name in item:
                    data[item] = data[item].astype(str)
                    data[item] = data[item].str.replace(',', '')
                    data[item] = pd.to_numeric(data[item], errors='coerce')
                    data[item] *= params[col_name]
            elif isinstance(col_name,tuple):
                for i in range(len(col_name)):
                    if col_name[i] in item:
                        data[item] = data[item].astype(str)
                        data[item] = data[item].str.replace(',', '')
                        data[item] = pd.to_numeric(data[item], errors='coerce')
                        data[item] *= params[col_name]
            else:
                print("params输入格式错误,错误的键为{}".format(item))

    return data



# 输入
---

In [9]:
# 动物
params = {
    ("Raw milk of cattle"):"Dairy cows",
    ("Meat of cattle with the bone, fresh or chilled","Meat of buffalo, fresh or chilled"):"Beef cattle",
    # "excl cattle":
    "Meat of pig with the bone, fresh or chilled":"Fattening pigs, live weight 50 kg or over",
    "Sheep and Goats":"Sheep",
    "Hen eggs in shell, fresh":"County Layers",
    "Meat of chickens, fresh or chilled":"County broiler"
} # fao种类与各个国家官网下载的种类一一对应,左边写fao的种类名称，右边写国家官网种类名称


fao_target_path = 'D:/中科院数据下载/eurostat/animal_geo_ok/FAO数据/' # 规范化后的fao数据的保存路径
fao_data_path =  'D:/中科院数据下载/eurostat/animal_geo_ok/FAO数据/animal_fao.csv'  # fao原始数据的存放路径
data_path = 'D:/中科院数据下载/eurostat/animal_geo_ok/' # 需要校对的国家数据存放路径
r_path = 'D:/中科院数据下载/eurostat/animal_geo_ok/FAO数据/比例县.csv' # 历史比例存放路径
data_fao_ok = 'D:/中科院数据下载/eurostat/动物_fao_ok/' # 校对完成后的数据存放路径

# 校对国家的开始与结束年份
year_begin = 1980
year_end = 2021

# 单位转化参数
params_unit = {
    ("Dairy cows","Beef cattle","excl cattle","Fattening pigs, live weight 50 kg or over","Sheep","County Layers","County broiler"):1000
}

In [31]:
# 农作物

params = {
    'Other stimulant, spice and aromatic crops, n.e.c.':'Harvested production in EU standard humidity (1000 t)_Aromatic, medicinal and culinary plants',
    'Barley':'Harvested production in EU standard humidity (1000 t)_Barley',
    'Other citrus fruit, n.e.c.':'Harvested production in EU standard humidity (1000 t)_Citrus fruits',
    'Wheat':'Harvested production in EU standard humidity (1000 t)_Common wheat and spelt',
    'Seed cotton, unginned':'Harvested production in EU standard humidity (1000 t)_Cotton seed',
    'Other pulses n.e.c.':'Harvested production in EU standard humidity (1000 t)_Dry pulses and protein crops for the production of grain (including seed and mixtures of cereals and pulses)',
    'Other fibre crops, raw, n.e.c.':'Harvested production in EU standard humidity (1000 t)_Fibre crops',
    'Other berries and fruits of the genus vaccinium n.e.c.':'Harvested production in EU standard humidity (1000 t)_Fruits, berries and nuts (excluding citrus fruits, grapes and strawberries)',
    'Green corn (maize)':'Harvested production in EU standard humidity (1000 t)_Green maize',
    'True hemp, raw or retted':'Harvested production in EU standard humidity (1000 t)_Hemp',
    'Linseed':'Harvested production in EU standard humidity (1000 t)_Linseed (oilflax)',
    'Olives':'Harvested production in EU standard humidity (1000 t)_Olives',
    'Other pulses n.e.c.':'Harvested production in EU standard humidity (1000 t)_Other dry pulses and protein crops n.e.c.',
    'Potatoes':'Harvested production in EU standard humidity (1000 t)_Potatoes (including seed potatoes)',
    'Sorghum':'Harvested production in EU standard humidity (1000 t)_Sorghum',
    'Soya beans':'Harvested production in EU standard humidity (1000 t)_Soya',
    'Barley':'Harvested production in EU standard humidity (1000 t)_Spring barley',
    'Sugar beet':'Harvested production in EU standard humidity (1000 t)_Sugar beet (excluding seed)',
    'Sunflower seed':'Harvested production in EU standard humidity (1000 t)_Sunflower seed',
    'Lupins':'Harvested production in EU standard humidity (1000 t)_Sweet lupins',
    'Unmanufactured tobacco':'Harvested production in EU standard humidity (1000 t)_Tobacco',
    'Triticale':'Harvested production in EU standard humidity (1000 t)_Triticale',
    'Rye':'Harvested production in EU standard humidity (1000 t)_Rye',
    'Oats':'Harvested production in EU standard humidity (1000 t)_Oats',
    'Rice':'Harvested production in EU standard humidity (1000 t)_Rice',
    'Peas, dry':'Harvested production in EU standard humidity (1000 t)_Field peas',
    'Other fibre crops, raw, n.e.c.':'Harvested production in EU standard humidity (1000 t)_Other fibre crops n.e.c.',
    'Other oil seeds, n.e.c.':'Harvested production in EU standard humidity (1000 t)_Other oilseed crops n.e.c.',
    'Green corn (maize)':'Harvested production in EU standard humidity (1000 t)_Grain maize and corn-cob-mix',
    'Grapes':'Harvested production in EU standard humidity (1000 t)_Grapes',
} # fao种类与各个国家官网下载的种类一一对应,左边写fao的种类名称，右边写国家官网种类名称


fao_target_path = 'D:/中科院数据下载/eurostat/农作物_ok/FAO/' # 规范化后的fao数据的保存路径
fao_data_path =  'D:/中科院数据下载/eurostat/农作物_ok/FAO/FAOSTAT_data_en_1-28-2024.csv'  # fao原始数据的存放路径
data_path = 'D:/中科院数据下载/eurostat/农作物_ok/' # 需要校对的国家数据存放路径
r_path = 'D:/中科院数据下载/eurostat/农作物_ok/FAO/比例县.csv' # 历史比例存放路径
data_fao_ok = 'D:/中科院数据下载/eurostat/农作物_fao_ok/' # 校对完成后的数据存放路径

# 校对国家的开始与结束年份
year_begin = 1975
year_end = 2022

# 单位转化参数
params_unit = {
    
}

# 执行
---

In [32]:
# 执行函数一
# 规范化fao数据
# fao_data = pd.read_csv(fao_data_path)
# fao_standard(fao_data,params,fao_target_path)

# 执行函数二
r = pd.DataFrame()

for y in range(year_begin,year_end+1):
    data = pd.read_csv(data_path+str(y)+".csv")
    
    data_fao = pd.read_csv(fao_target_path+str(y)+".csv")
    r_tmp = cal_r(data,data_fao,data_country_colname="country_label",data_fao_item_colname="Item")

    r = pd.concat([r,r_tmp],axis=0)

# 执行函数三
# 进行校对
# 与FAO数据校对
for y in range(year_begin,year_end+1):
    data = pd.read_csv(data_path+str(y)+".csv")
    data = unit_conversion(data,params_unit) # 单位转化
    data_fao = pd.read_csv(fao_target_path+str(y)+".csv")
    data_ok = single_proofread_r(data,data_fao,r,data_country_colname="country_label",data_fao_item_colname="Item")

    data_ok.to_csv(data_fao_ok+str(y)+".csv",index=False,encoding="utf-8-sig")




In [1]:
# 统一列名称
def rename_become_faoname(data,params):
    # 构建一个替换映射字典，其中每个旧列名映射到新列名  
    replacement_mapping = {}  
    for new_names, old_name in params.items():  
        if isinstance(new_names, tuple):  
            # 如果新名字是一个元组，那么我们合并它们为一个长字符串  
            replacement_mapping[old_name] = ", ".join(new_names)  
        else:  
            # 如果不是元组，那么直接赋值  
            replacement_mapping[old_name] = new_names 
 
    # 使用映射字典替换列名  
    return data.rename(columns=replacement_mapping)  

In [14]:
import pandas as pd 
# 巴西农作物

params = {
    "Pineapples":"Pineapple*",
    "Avocados":"Abacate",
    "Seed cotton, unginned":"Herbaceous cotton (seed)",
    "Green garlic":"Garlic",
    "Groundnuts, excluding shelled":"Peanuts (in shell)",
    "Rice":"Rice",
    "Oats":"Oats (grain)",
    "Olives":"Olive",
    "Bananas":"Banana (bunch)",
    "Sweet potatoes":"Sweet potatoes",
    "potatos":"Batata-inglesa",
    "Natural rubber in primary forms":"Rubber (coagulated latex)",
    "Cocoa beans":"Cocoa beans",
    "Coffee, green":"Coffee (beans) Total",
    "Sugar cane":"Sugarcane",
    "Persimmons":"Persimmon",
    "Cashew nuts, in shell":"Cashew nuts",
    "Onions and shallots, dry (excluding dehydrated)":"Onion",
    "Rye":"Rye (grain)",
    "Barley":"Barley (grain)",
    "Tea leaves":"Tea(leafy green)",
    "Oil palm fruit":"Palm oil(coconut bunch)",
    "Maté leaves":"Yerba mate (green leaf)",
    "Peas, dry":"Peas (grain)",
    "Broad beans and horse beans, dry":"Fava beans",
    "Beans, dry":"Beans (beans)",
    "Figs":"Fig",
    "Unmanufactured tobacco":"Tobacco",
    "Sunflower seed":"Sunflower (grain)",
    "Jute, raw or retted":"Jute (fiber)",
    "Oranges":"Orange",
    "Lemons and limes":"Lemon",
    "Linseed":"Flax (seed)",
    "Apples":"Apple",
    "Kenaf, and other textile bast fibres, raw or retted":"Mallow (fiber)",
    "Papayas":"Papaya",
    "Castor oil seeds":"Mormon(bag)",
    "Cassava, fresh":"Cassava",
    "Quinces":"Marble",
    "Watermelons":"Watermelon",
    "Cantaloupes and other melons":"Melon",
    "Maize (corn)":"Corn (in grain)",
    "Walnuts, in shell":"Walnut(dried fruit)",
    "Oil palm fruit":"Palm",
    "Pears":"Pear",
    "Peaches and nectarines":"Peach",
    "Pepper (Piper spp.), raw":"Black pepper",
    "Ramie, raw or retted":"Rami (fiber)",
    "Sisal, raw":"Sisal or agave (fiber)",
    "Soya beans":"Soybeans",
    "Sorghum":"Sorghum (grain)",
    "Tangerines, mandarins, clementines":"Tangerine",
    "Tomatoes":"Tomato",
    "Wheat":"Wheat (grain)",
    "Triticale":"Triticale (grain)",
    "Tung nuts":"Tungue(fruto seco)",
    "Grapes":"Grape"
}

# 校对国家的开始与结束年份

year_begin = 2022
year_end = 2022
path = '校对总量全（新）/巴西/巴西农作物_fao_ok/'
for y in range(year_begin,year_end+1):
    data = pd.read_csv(path+str(y)+".csv",encoding='ISO-8859-1')
    D = rename_become_faoname(data,params)
    D.to_csv(path+str(y)+".csv",index=False,encoding='ISO-8859-1')


In [ ]:
import pandas as pd
params = {
    "Pineapples":"Pineapple*",
    "Avocados":"Abacate",
    "Seed cotton, unginned":"Herbaceous cotton (seed)",
    "Green garlic":"Garlic",
    "Groundnuts, excluding shelled":"Peanuts (in shell)",
    "Rice":"Rice (paddy)",
    "Oats":"Oats (grains)",
    "Olives":"Olive",
    "Bananas":"Banana (cacho)",
    "Sweet potatoes":"Sweet potato",
    "potatos":"Batata-inglesa",
    "Natural rubber in primary forms":"Rubber (coagulated latex)",
    "Cocoa beans":"Cocoa (in almonds)",
    "Coffee, green":"Coffee beans Total",
    "Sugar cane":"Sugar cane",
    "Persimmons":"Persimmon",
    "Cashew nuts, in shell":"Cashew nuts",
    "Onions and shallots, dry (excluding dehydrated)":"Onion",
    "Rye":"Rye (grain)",
    "Barley":"Barley (grain)",
    "Tea leaves":"Tea (leafy green)",
    "Oil palm fruit":"Palm oil (coconut bunch)",
    "Maté leaves":"Yerba mate (leafy green)",
    "Peas, dry":"Peas (grains)",
    "Broad beans and horse beans, dry":"Fava beans (grain)",
    "Beans, dry":"Beans (beans)",
    "Figs":"Fig",
       "Unmanufactured tobacco":"Tobacco (leaf)",
    "Sunflower seed":"Sunflower (grain)",
    "Jute, raw or retted":"Jute (fiber)",
    "Oranges":"Orange",
    "Lemons and limes":"Lemon",
    "Linseed":"Flax (seed)",
    "Apples":"Apple",
    "Kenaf, and other textile bast fibres, raw or retted":"Mallow (fiber)",
    "Papayas":"Papaya",
    "Castor oil seeds":"Mormon (bag)",
    "Cassava, fresh":"Cassava",
    "Quinces":"Marble",
    "Watermelons":"Watermelon",
    "Cantaloupes and other melons":"Melon",
    "Maize (corn)":"Corn (grain)",
    "Walnuts, in shell":"Walnut (dried fruit)",
    "Oil palm fruit":"Palm",
    "Pears":"Pear",
    "Peaches and nectarines":"Peach",
    "Pepper (Piper spp.), raw":"Black pepper",
    "Ramie, raw or retted":"Rami (fiber)",
    "Sisal, raw":"Sisal or agave (fiber)",
    "Soya beans":"Soybeans",
    "Sorghum":"Sorghum (grain)",
    "Tangerines, mandarins, clementines":"Tangerine",
    "Tomatoes":"Tomato",
    "Wheat":"Wheat (grain)",
    "Triticale":"Triticale (grain)",
    "Tung nuts":"Tungue (fruto seco)",
    "Grapes":"Grape"
}
year_begin = 2022
year_end = 2022
path = '校对总量全（新）/巴西/农作物_fao_ok/'
for y in range(year_begin,year_end+1):
    data = pd.read_csv(path+str(y)+".csv")
    D = rename_become_faoname(data,params)
    D.to_csv(path+str(y)+".csv",index=False,encoding="utf-8-sig")

In [2]:
import pandas as pd
# 欧盟农作物
params = {
    'Other stimulant, spice and aromatic crops, n.e.c.':'Harvested production (1000 t)_Aromatic, medicinal and culinary plants',
    'Barley':'Harvested production (1000 t)_Barley',
    'Oranges':'Harvested production (1000 t)_Citrus fruits',
    'Wheat':'Harvested production (1000 t)_Wheat and spelt',
    'Seed cotton, unginned':'Harvested production (1000 t)_Cotton seed',
    
    'True hemp, raw or retted':'Harvested production (1000 t)_Hemp',
    'Linseed':'Harvested production (1000 t)_Linseed (oilflax)',
    'Olives':'Harvested production (1000 t)_Olives',
    'Potatoes':'Harvested production (1000 t)_Potatoes (including seed potatoes)',
    'Sorghum':'Harvested production (1000 t)_Sorghum',
    'Soya beans':'Harvested production (1000 t)_Soya',
    'Sugar beet':'Harvested production (1000 t)_Sugar beet (excluding seed)',
    'Sunflower seed':'Harvested production (1000 t)_Sunflower seed',
    'Lupins':'Harvested production (1000 t)_Sweet lupins',
    'Unmanufactured tobacco':'Harvested production (1000 t)_Tobacco',
    'Triticale':'Harvested production (1000 t)_Triticale',
    'Rye':'Harvested production (1000 t)_Rye',
    'Oats':'Harvested production (1000 t)_Oats',
    'Rice':'Harvested production (1000 t)_Rice',
    'Peas, dry':'Harvested production (1000 t)_Field peas',
    'Other fibre crops, raw, n.e.c.':'Harvested production (1000 t)_Other fibre crops n.e.c.',
    'Other oil seeds, n.e.c.':'Harvested production (1000 t)_Other oilseed crops n.e.c.',
    'Grapes':'Harvested production (1000 t)_Grapes',
    
    'Broad beans and horse beans, dry':'Harvested production (1000 t)_Broad and field beans',
    'Flax, raw or retted':'Harvested production (1000 t)_Fibre crops',
    
    ('Watermelons','Strawberries'):'Harvested production (1000 t)_Fruits, berries and nuts (excluding citrus fruits, grapes and strawberries)',  #  Strawberries
    'Maize (corn)':'Harvested production (1000 t)_Green maize',
    'Hop cones':'Harvested production (1000 t)_Hops',
    'Other beans, green':'Harvested production (1000 t)_Leguminous plants harvested green',
    
    'Millet':'Harvested production (1000 t)_Other cereals n.e.c. (buckwheat, millet, canary seed, etc.)',
    'Rape or colza seed':'Harvested production (1000 t)_Rape, turnip rape, sunflower seeds and soya',
    'Cereals n.e.c.':'Harvested production (1000 t)_Spring cereal mixtures (mixed grain other than maslin)'
} # fao种类与各个国家官网下载的种类一一对应,左边写fao的种类名称，右边写国家官网种类名称

import chardet
year_begin = 1980
year_end = 1980
path = '校对总量全（新）/欧盟/'
for y in range(year_begin,year_end+1):
    with open(path+str(y)+".csv", 'rb') as file:  
            result = chardet.detect(file.read())  # 或者file.read(10000)来读取部分文件进行检测  
    encoding = result['encoding']  
    data = pd.read_csv(path+str(y)+".csv",encoding=encoding)
    D = rename_become_faoname(data,params)
    D.to_csv(path+str(y)+"FaoName.csv",index=False,encoding="utf-8-sig")

In [3]:
import pandas as pd
# 欧盟农作物
params = {
    'Other stimulant, spice and aromatic crops, n.e.c.':'Harvested production in EU standard humidity (1000 t)_Aromatic, medicinal and culinary plants',
    'Barley':'Harvested production in EU standard humidity (1000 t)_Barley',
    'Oranges':'Harvested production in EU standard humidity (1000 t)_Citrus fruits',
    'Wheat':'Harvested production in EU standard humidity (1000 t)_Wheat and spelt',
    'Seed cotton, unginned':'Harvested production in EU standard humidity (1000 t)_Cotton seed',
    
    'True hemp, raw or retted':'Harvested production in EU standard humidity (1000 t)_Hemp',
    'Linseed':'Harvested production in EU standard humidity (1000 t)_Linseed (oilflax)',
    'Olives':'Harvested production in EU standard humidity (1000 t)_Olives',
    'Potatoes':'Harvested production in EU standard humidity (1000 t)_Potatoes (including seed potatoes)',
    'Sorghum':'Harvested production in EU standard humidity (1000 t)_Sorghum',
    'Soya beans':'Harvested production in EU standard humidity (1000 t)_Soya',
    'Sugar beet':'Harvested production in EU standard humidity (1000 t)_Sugar beet (excluding seed)',
    'Sunflower seed':'Harvested production in EU standard humidity (1000 t)_Sunflower seed',
    'Lupins':'Harvested production in EU standard humidity (1000 t)_Sweet lupins',
    'Unmanufactured tobacco':'Harvested production in EU standard humidity (1000 t)_Tobacco',
    'Triticale':'Harvested production in EU standard humidity (1000 t)_Triticale',
    'Rye':'Harvested production in EU standard humidity (1000 t)_Rye',
    'Oats':'Harvested production in EU standard humidity (1000 t)_Oats',
    'Rice':'Harvested production in EU standard humidity (1000 t)_Rice',
    'Peas, dry':'Harvested production in EU standard humidity (1000 t)_Field peas',
    'Other fibre crops, raw, n.e.c.':'Harvested production in EU standard humidity (1000 t)_Other fibre crops n.e.c.',
    'Other oil seeds, n.e.c.':'Harvested production in EU standard humidity (1000 t)_Other oilseed crops n.e.c.',
    'Grapes':'Harvested production in EU standard humidity (1000 t)_Grapes',
    
    'Broad beans and horse beans, dry':'Harvested production in EU standard humidity (1000 t)_Broad and field beans',
    'Flax, raw or retted':'Harvested production in EU standard humidity (1000 t)_Fibre crops',
    
    ('Watermelons','Strawberries'):'Harvested production in EU standard humidity (1000 t)_Fruits, berries and nuts (excluding citrus fruits, grapes and strawberries)',
    'Maize (corn)':'Harvested production in EU standard humidity (1000 t)_Green maize',
    'Hop cones':'Harvested production in EU standard humidity (1000 t)_Hops',
    'Other beans, green':'Harvested production in EU standard humidity (1000 t)_Leguminous plants harvested green',
   
    'Millet':'Harvested production in EU standard humidity (1000 t)_Other cereals n.e.c. (buckwheat, millet, canary seed, etc.)',
    'Rape or colza seed':'Harvested production in EU standard humidity (1000 t)_Rape, turnip rape, sunflower seeds and soya',
    'Cereals n.e.c.':'Harvested production in EU standard humidity (1000 t)_Spring cereal mixtures (mixed grain other than maslin)'
} # fao种类与各个国家官网下载的种类一一对应,左边写fao的种类名称，右边写国家官网种类名称

import chardet
year_begin = 2021
year_end = 2021
path = '校对总量全（新）/欧盟/'
for y in range(year_begin,year_end+1):
    with open(path+str(y)+".csv", 'rb') as file:  
            result = chardet.detect(file.read())  # 或者file.read(10000)来读取部分文件进行检测  
    encoding = result['encoding']  
    data = pd.read_csv(path+str(y)+".csv",encoding=encoding)
    D = rename_become_faoname(data,params)
    D.to_csv(path+str(y)+"FaoName.csv",index=False,encoding="utf-8-sig")

In [46]:
import pandas as pd
# 欧盟动物

params = {
    ("Raw milk of cattle"):"Dairy cows",
    ("Meat of cattle with the bone, fresh or chilled","Meat of buffalo, fresh or chilled"):"Beef cattle",
    # "excl cattle":
    "Meat of pig with the bone, fresh or chilled":"Fattening pigs, live weight 50 kg or over",
    "Sheep and Goats":"Sheep",
    "Hen eggs in shell, fresh":"County Layers",
    "Meat of chickens, fresh or chilled":"County broiler"
} # fao种类与各个国家官网下载的种类一一对应,左边写fao的种类名称，右边写国家官网种类名称



year_begin = 1980
year_end = 2021
path = '校对总量全（新）/欧盟/动物_fao_ok/'
for y in range(year_begin,year_end+1):
    data = pd.read_csv(path+str(y)+".csv")
    D = rename_become_faoname(data,params)
    D.to_csv(path+str(y)+".csv",index=False,encoding="utf-8-sig")

In [49]:
import pandas as pd
# 美国农作物
params = {
    'Barley':'PRODUCTION_BARLEY',
    'Beans, dry':'PRODUCTION_BEANS',
    'Rape or colza seed':'PRODUCTION_CANOLA',
    'Chick peas, dry':'PRODUCTION_CHICKPEAS',
    'Green corn (maize)':'PRODUCTION_CORN',
    'Seed cotton, unginned':'PRODUCTION_COTTON',
    'Flax, processed but not spun':'PRODUCTION_FLAXSEED',
    'Lentils, dry':'PRODUCTION_LENTILS',
    'Mustard seed':'PRODUCTION_MUSTARD',
    'Oats':'PRODUCTION_OATS',
    'Groundnuts, excluding shelled':'PRODUCTION_PEANUTS',
    'Peas, dry':'PRODUCTION_PEAS',
    'Rice':'PRODUCTION_RICE',
    'Rye':'PRODUCTION_RYE',
    'Safflower seed':'PRODUCTION_SAFFLOWER',
    'Sorghum':'PRODUCTION_SORGHUM',
    'Soya beans':'PRODUCTION_SOYBEANS',
    'Sugar beet':'PRODUCTION_SUGARBEETS',
    'Sugar cane':'PRODUCTION_SUGARCANE',
    'Sunflower seed':'PRODUCTION_SUNFLOWER',
    'Unmanufactured tobacco':'PRODUCTION_TOBACCO',
    'Wheat':'PRODUCTION_WHEAT',
    
} # fao种类与各个国家官网下载的种类一一对应,每个种类都要写,本质是改名，统一名称



year_begin = 1961
year_end = 2021
path = '校对总量全（新）/美国/农作物_fao_ok/'
for y in range(year_begin,year_end+1):
    data = pd.read_csv(path+str(y)+".csv")
    D = rename_become_faoname(data,params)
    D.to_csv(path+str(y)+".csv",index=False,encoding="utf-8-sig")

In [50]:
import pandas as pd
# 巴西动物
params = {
    "Raw milk of cattle":"milk_cattle",
    "Meat of cattle with the bone, fresh or chilled":"beef_cattle",
    # "excl cattle":
    "Meat of pig with the bone, fresh or chilled":"Pork - total",
    ('Sheep','Goats'):"Sheep_Goat",
    "Hen eggs in shell, fresh":"Layers",
    "Meat of chickens, fresh or chilled":"Broilers",
    "Horse meat, fresh or chilled":'Horse'
} 
 # fao种类与各个国家官网下载的种类一一对应,左边写fao的种类名称，右边写国家官网种类名称



year_begin = 1974
year_end = 2022
path = '校对总量全（新）/巴西/动物_fao_ok/'
for y in range(year_begin,year_end+1):
    data = pd.read_csv(path+str(y)+".csv")
    D = rename_become_faoname(data,params)
    D.to_csv(path+str(y)+".csv",index=False,encoding="utf-8-sig")

In [52]:
import pandas as pd
# 澳大利亚农作物
params = {
    'Barley':'Cereal crops - Barley for grain - Production (t)',
    'Rape or colza seed':'Other crops - Oilseeds - Canola - Production (t)',
    'Chick peas, dry':'Other crops - Pulses and legumes - Chickpeas - Production (t)',
    'Lentils, dry':'Other crops - Pulses and legumes - Lentils - Production (t)',
    'Oats':'Cereal crops - Oats for grain - Production (t)',
    'Groundnuts, excluding shelled':'PRODUCTION_PEANUTS',
    'Rice':'Cereal crops - Rice for grain - Production (t)',
    'Sorghum':'Cereal crops - Sorghum for grain - Production (t)',
    'Sugar cane':'Other crops - Sugar cane - Cut for crushing - Production (t)',
    'Wheat':'Cereal crops - Wheat for grain - Production (t)',
    'Maize (corn)':'Cereal crops - Maize for grain - Production (t)',
    'Lupins':'Other crops - Pulses and legumes - Lupins - Production (t)',
    'Tangerines, mandarins, clementines':'Fruit and nuts - Citrus fruit - Mandarins - Production (t)',
    'Oranges':'Fruit and nuts - Citrus fruit - Oranges - Production (t)',
    'Cherries':'Fruit and nuts - Stone fruit - Cherries - Production (t)',
    'Apples':'Fruit and nuts - Other orchard fruit - Apples - Production (t)',
    'Avocados':'Fruit and nuts - Other orchard fruit - Avocados - Production (t)',
    'Mangoes, guavas and mangosteens':'Fruit and nuts - Other orchard fruit - Mangoes - Production (t)',
    'Olives':'Fruit and nuts - Other orchard fruit - Olives - Production (t)',
    'Pears':'Fruit and nuts - Other orchard fruit - Pears (including Nashi) - Production (t)',
    'Almonds, in shell':'Fruit and nuts - Nuts - Almonds - Production (t) (f)',
    'Other nuts (excluding wild edible nuts and groundnuts), in shell, n.e.c.':'Fruit and nuts - Nuts - Macadamias - Production (t) (g)',
    'Strawberries':'Fruit and nuts - Berry fruit - Strawberries - Production (t)',
    'Bananas':'Fruit and nuts - Plantation fruit - Bananas - Production (t)',
    'Pineapples':'Fruit and nuts - Plantation fruit - Pineapples - Production (t)',
    'Grapes':'Fruit and nuts - Grapes - Total - Production (t)',
    'Other beans, green':'Vegetables - Beans (including french and runner) - Production (kg)',
    'Cabbages':'Vegetables - Cabbages - Production (t)',
    'Chillies and peppers, green (Capsicum spp. and Pimenta spp.)':'Vegetables - Capsicums (excluding chillies) - Production (kg)',
    'Carrots and turnips':'Vegetables - Carrots - Production (t)',
    'Cucumbers and gherkins':'Vegetables - Cucumbers - Production (t)',
    'Lettuce and chicory':'Vegetables - Lettuces - Production (kg)',
    'Cantaloupes and other melons':'Vegetables - Melons - Production (t) (i)',
    'Mushrooms and truffles':'Vegetables - Mushrooms - Production (kg)',
    'Onions and shallots, dry (excluding dehydrated)':'Vegetables - Onions - Production (t)',
    'Potatoes':'Vegetables - Potatoes - Production (t)',
    'Pumpkins, squash and gourds':'Vegetables - Pumpkins - Production (t)',
    'Green corn (maize)':'Vegetables - Sweet corn - Production (t)',
    'Tomatoes':'Vegetables - Tomatoes - Production (t)'
} # fao种类与各个国家官网下载的种类一一对应,每个种类都要写,本质是改名，统一名称
# 左边fao，右边官网
 # fao种类与各个国家官网下载的种类一一对应,左边写fao的种类名称，右边写国家官网种类名称



year_begin = 2021
year_end = 2021
path = '校对总量全（新）/澳大利亚/农作物_fao_ok/'
for y in range(year_begin,year_end+1):
    data = pd.read_csv(path+str(y)+".csv")
    D = rename_become_faoname(data,params)
    D.to_csv(path+str(y)+".csv",index=False,encoding="utf-8-sig")

In [57]:
import pandas as pd
# 美国动物
params = {
    "Raw milk of cattle":"cattle_cow_milk",
    ("Meat of cattle with the bone, fresh or chilled","Meat of buffalo, fresh or chilled"):"cattle_cow_beef",
    # "excl cattle":
    "Meat of pig with the bone, fresh or chilled":"hogs",
    ("Sheep","Goats"):"sheep_goats",
    "Hen eggs in shell, fresh":"layers",
    "Meat of chickens, fresh or chilled":"broilers"
}

 # fao种类与各个国家官网下载的种类一一对应,左边写fao的种类名称，右边写国家官网种类名称



year_begin = 2017
year_end = 2017
path = '校对总量全（新）/美国/动物_fao_ok/'
for y in range(year_begin,year_end+1):
    data = pd.read_csv(path+str(y)+".csv")
    D = rename_become_faoname(data,params)
    D.to_csv(path+str(y)+".csv",index=False,encoding="utf-8-sig")